# Entrenamiento de un modelo YOLO para detectar el tubo de muestras (PAROL6)

Entrena `yolo26n` sobre el dataset sintetico generado por
`synthetic_dataset_generator_v2.py` (imagenes monocromas de la NavCam, una sola
clase: `mars2020tube`). Pensado para ejecutarse en Google Colab **de arriba a
abajo**.

**Orden del notebook**
1. Instalar dependencias
2. Montar Google Drive
3. Apuntar al dataset (`data.yaml`)
4. Configurar Weights & Biases
5. Inyectar el aumento sim2real (ruido, compresion, motas de polvo)
6. Optimizar hiperparametros (W&B Sweep bayesiano)
7. Entrenamiento final con los hiperparametros ganadores
8. Curvas de entrenamiento
9. Evaluacion (validacion y test)
10. Predicciones de ejemplo
11. Descargar resultados

## 1. Instalar dependencias

`ultralytics` (YOLO), `wandb` (seguimiento y sweeps) y `albumentations` (aumento
sim2real). Si Colab pide reiniciar el entorno tras instalar, hazlo y vuelve a
ejecutar desde esta celda.

In [ ]:
# albumentations habilita el aumento sim2real; Ultralytics lo aplica
# automaticamente si la libreria esta presente.
!pip install -q wandb albumentations
!pip install -q ultralytics

## 2. Montar Google Drive

El dataset vive en Drive. Lo montamos para que Colab pueda leerlo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Apuntar al dataset

`synthetic_dataset_generator_v2.py` ya genera el dataset YOLO con su `data.yaml`:

```
parol6_synth_dataset/
├── data.yaml
├── images/{train,val,test}/img_XXXXXX.png   # gris (3 canales replicados)
└── labels/{train,val,test}/img_XXXXXX.txt   # YOLO: clase cx cy w h (bbox visible)
                                             # .txt vacio = negativo
```

`data.yaml` declara `nc: 1` y `names: ['mars2020tube']`. Como se genero en otra
maquina, reescribimos su campo `path:` al directorio real en Drive (idempotente)
para que Ultralytics resuelva `train/val/test` en Colab.

In [ ]:
import os
import yaml

data_yaml_path = "/content/drive/MyDrive/Ciencias Espaciales/TFM/parol6_synth_dataset_v3/data.yaml"
DATASET_ROOT = os.path.dirname(data_yaml_path)

with open(data_yaml_path) as f:
    _data_cfg = yaml.safe_load(f)
_data_cfg["path"] = DATASET_ROOT
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(_data_cfg, f, sort_keys=False)

print("data.yaml:", data_yaml_path)
print(_data_cfg)

## 4. Weights & Biases

W&B registra metricas y orquesta el sweep de hiperparametros. Necesitas una
cuenta gratuita en [wandb.ai](https://wandb.ai/); al ejecutar `wandb.login()`
pega tu API key cuando se solicite.

In [ ]:
!yolo settings wandb=True

In [ ]:
import wandb

wandb.login()

## 5. Aumento sim2real: simulacion de polvo marciano (Albumentations)

El dataset se genera en gris "limpio" a proposito: el ruido de sensor, la
compresion JPEG, las motas de polvo y el desenfoque se simulan **aqui**, en el
aumento de entrenamiento, para cerrar el gap sim2real con la NavCam real.

Ultralytics integra Albumentations automaticamente, pero solo con un set fijo.
La celda siguiente **inyecta transforms adicionales** parcheando la clase
`Albumentations` de Ultralytics. Se ejecuta una vez y afecta a TODO
entrenamiento posterior (sweep incluido), de modo que los hiperparametros que
encuentres transfieran al entrenamiento final.

> Las imagenes son monocromas: no se tocan tono ni saturacion (`hsv_h=0`,
> `hsv_s=0` en `train`); solo brillo y degradaciones fisicas. Los parametros
> estan fijados a mano porque los valores por defecto de Albumentations 2.x no
> sirven aqui: `GaussNoise` usa sigma del 20-44 % e independiente por canal
> (rompe el gris replicado), `ImageCompression` calidad 99-100 (sin efecto) y
> `CoarseDropout` 1-2 huecos del 10-20 % del lado, capaces de tapar el tubo
> entero. Ultralytics no ajusta las cajas con estas transforms (se aplican solo
> a la imagen), asi que los huecos deben ser pequenios frente al tubo.
> `RandomFog` se descarta: aplica siempre un desenfoque gaussiano de
> min(alto, ancho)/30 px (~21 px a 640), mas ancho que el tubo.

In [ ]:
# --- Inyeccion de aumento sim2real en el pipeline de Ultralytics ---
# Parchea ultralytics.data.augment.Albumentations para aniadir transforms que
# imitan las degradaciones de la NavCam real sobre el render sintetico.
# Robusto a cambios de API: cada transform se construye en un try/except y se
# omite si esta version de albumentations no lo soporta.
import albumentations as A
import ultralytics.data.augment as _aug


def _build_dust_transforms():
    """Transforms de degradacion fisica, validas en imagenes monocromas."""
    candidates = [
        # Ruido de sensor: sigma 1-4 % del rango (2.5-10 niveles de gris), el
        # MISMO en los 3 canales para no romper el gris replicado.
        ("GaussNoise",               lambda: A.GaussNoise(std_range=(0.01, 0.04),
                                                          per_channel=False, p=0.30)),
        ("ISONoise",                 lambda: A.ISONoise(p=0.20)),  # en gris no altera el color
        ("RandomBrightnessContrast", lambda: A.RandomBrightnessContrast(
                                         brightness_limit=0.25,
                                         contrast_limit=0.25, p=0.50)),        # exposicion
        ("RandomGamma",              lambda: A.RandomGamma(p=0.20)),
        ("MotionBlur",               lambda: A.MotionBlur(blur_limit=7, p=0.20)),
        ("Blur",                     lambda: A.Blur(blur_limit=5, p=0.10)),
        # Motas de polvo: 4-12 huecos del 1-3 % del lado (6-19 px a 640). Las
        # cajas no se ajustan, asi que deben ser pequenios frente al tubo.
        ("CoarseDropout",            lambda: A.CoarseDropout(num_holes_range=(4, 12),
                                                             hole_height_range=(0.01, 0.03),
                                                             hole_width_range=(0.01, 0.03),
                                                             fill=0, p=0.20)),
        # Compresion JPEG del enlace de bajada (por defecto 99-100: sin efecto)
        ("ImageCompression",         lambda: A.ImageCompression(quality_range=(50, 90), p=0.20)),
    ]
    out = []
    for name, factory in candidates:
        try:
            out.append(factory())
        except Exception as exc:
            print(f"  warning: '{name}' omitido (incompatible): {exc}")
    return out


# Respalda el __init__ ORIGINAL una sola vez. Asi podemos re-ejecutar la celda
# sin apilar parches: siempre envolvemos el original, nunca uno ya parcheado.
if not hasattr(_aug.Albumentations, "_orig_init_backup"):
    _aug.Albumentations._orig_init_backup = _aug.Albumentations.__init__
_ORIG_ALBU_INIT = _aug.Albumentations._orig_init_backup


# *args/**kwargs: en esta version Ultralytics llama Albumentations(p=1.0,
# transforms=...); aceptamos cualquier firma y la reenviamos al original.
def _patched_albu_init(self, *args, **kwargs):
    _ORIG_ALBU_INIT(self, *args, **kwargs)   # set por defecto de Ultralytics
    try:
        extra = _build_dust_transforms()
        if not extra:
            return
        base_compose = self.transform
        base_T = list(base_compose.transforms) if base_compose is not None else []
        # Preservar la config de bboxes del Compose original. En esta version de
        # Ultralytics es None: las transforms se aplican SOLO a la imagen, y la
        # llamada es self.transform(image=...) sin bboxes ni class_labels. Si
        # forzamos bbox_params/label_fields propios, albumentations falla con
        # "label_fields are not valid". En versiones antiguas, en cambio, trae
        # label_fields y hay que respetarla.
        bbox_params = None
        try:
            proc = getattr(base_compose, "processors", {}).get("bboxes")
            if proc is not None:
                bbox_params = proc.params
        except Exception:
            bbox_params = None
        # Todas las transforms aniadidas son pixel-level: no alteran las bboxes.
        self.transform = A.Compose(base_T + extra, bbox_params=bbox_params)
        print("OK aumento sim2real inyectado:",
              ", ".join(t.__class__.__name__ for t in extra))
    except Exception as exc:
        print(f"warning: no se pudo inyectar el aumento sim2real: {exc}")


_aug.Albumentations.__init__ = _patched_albu_init
print("Parche de aumento listo. Se aplicara en el proximo model.train().")

## 6. Optimizacion de hiperparametros (W&B Sweep bayesiano)

Busqueda bayesiana de `lr0`, `lrf`, `momentum`, `weight_decay` y `batch`,
maximizando `mAP50-95` en validacion.

**Claves para que sea rapido en Colab** (si no, son horas):
- `cache=True`: las imagenes (~300 MB) se cargan a RAM en la 1a epoca y dejan de
  leerse desde Google Drive. El FUSE de Drive es el cuello de botella real
  (relee cada epoca con `cache=False`). Con un dataset tan pequenio es seguro.
- `fraction=0.3` + `epochs=40` + `patience=8`: para *rankear* hiperparametros no
  hace falta entrenar a fondo; basta una pasada corta sobre parte del train. El
  entrenamiento a fondo se reserva para la seccion 7.
- **No usamos Hyperband de W&B**: su corte temprano solo *marca* el run para
  parar, pero Ultralytics no consulta esa senial y seguiria hasta el final. La
  parada temprana EFECTIVA es la interna de Ultralytics (`patience`).

`optimizer="SGD"` sigue siendo imprescindible (con `auto` se ignoran lr0/momentum).

Al terminar, abre el sweep en wandb.ai, coge el run que maximiza `mAP50-95(B)` y
copia sus hiperparametros al **entrenamiento final** (seccion 7).

In [ ]:
import gc
import torch
import wandb
from ultralytics import YOLO

PROJECT = "yolo_sweep_bayes"

sweep_config = {
    "method": "bayes",
    "metric": {"name": "metrics/mAP50-95(B)", "goal": "maximize"},
    "parameters": {
        "lr0":          {"distribution": "log_uniform_values", "min": 0.001,  "max": 0.02},
        "lrf":          {"distribution": "log_uniform_values", "min": 0.01,   "max": 0.2},
        "momentum":     {"distribution": "uniform",            "min": 0.7,    "max": 0.98},
        "weight_decay": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.001},
        "batch":        {"values": [16, 32]},
    },
}


def train_sweep():
    # El agente inyecta los hiperparametros muestreados en wandb.config.
    # NO pasar config=sweep_config aqui: sobrescribiria esos valores.
    with wandb.init() as run:
        config = wandb.config
        model = YOLO("yolo26n.pt")   # mismo punto de partida en cada prueba

        print(f"\nParametros de esta prueba: {dict(config)}")
        model.train(
            data=data_yaml_path,
            # --- presupuesto de BUSQUEDA (corto, solo para rankear) ---
            epochs=40,             # suficiente para comparar; patience corta antes
            fraction=0.3,          # 30% del train -> epocas mas rapidas
            patience=8,            # parada temprana interna (la que SI funciona)
            imgsz=640,
            optimizer="SGD",       # imprescindible para que lr0/momentum cuenten
            batch=config.batch,
            lr0=config.lr0,
            lrf=config.lrf,
            momentum=config.momentum,
            weight_decay=config.weight_decay,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.4,   # monocromo
            name=f"sweep_run_{run.name}",
            device=0,
            val=True,
            seed=42,
            workers=2,
            cache=True,            # ~300 MB a RAM: evita releer de Drive cada epoca
        )
        print(f"Prueba {run.name} finalizada.")

        # Liberar memoria entre pruebas
        del model
        gc.collect()
        torch.cuda.empty_cache()


sweep_id = wandb.sweep(sweep_config, project=PROJECT)
print(f"Lanzando agente para el sweep: {sweep_id}")
wandb.agent(sweep_id, function=train_sweep, count=15)

## 7. Entrenamiento final

Con los hiperparametros ganadores del sweep. Sustituir los valores de abajo
por los del mejor run antes de lanzar.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

# IMPORTANTE: optimizer="SGD" es obligatorio para que lr0/lrf/momentum se
# respeten. Con optimizer="auto" (por defecto) Ultralytics IGNORA lr0 y momentum.
results = model.train(
    data=data_yaml_path,
    epochs=350,            # presupuesto amplio
    patience=50,           # tolerancia alta antes de la parada temprana
    imgsz=640,

    # --- HIPERPARAMETROS GANADORES DEL SWEEP (sustituir por los reales) ---
    optimizer="SGD",
    batch=16,
    lr0=0.0159239543701858,
    lrf=0.0971806313337835,
    momentum=0.741334513916325,
    weight_decay=0.000127484785819259,
    cos_lr=True,           # schedule coseno: ayuda en entrenamientos largos

    # --- AUMENTO: imagenes monocromas -> sin tono/saturacion, solo brillo ---
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.4,
    # (ruido, compresion y motas de polvo los aniade el parche de la seccion 5)

    # --- VELOCIDAD: cachear a RAM evita releer de Drive cada epoca ---
    workers=2,
    cache=True,

    project=os.path.join(os.path.dirname(DATASET_ROOT), "runs"),
    name="yolo_tubo_v3_final",
    device=0,
)

## 8. Curvas de entrenamiento

Ultralytics guarda `results.png` (loss, precision, recall, mAP por epoca) en el
directorio del run. Tambien se sincroniza con W&B si iniciaste sesion.

In [ ]:
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
RUN_DIR = os.path.join(os.path.dirname(DATASET_ROOT), "runs", "yolo_tubo_v3_final")
model = YOLO(os.path.join(RUN_DIR, "weights", "best.pt"))

results_png = os.path.join(RUN_DIR, "results.png")
print("Curvas:", results_png)

plt.figure(figsize=(15, 8))
plt.imshow(plt.imread(results_png))
plt.axis("off")
plt.show()

## 9. Evaluacion

mAP en validacion. Tras `train`, `model` ya tiene los mejores pesos (`best.pt`).

In [ ]:
from ultralytics import YOLO
RUN_DIR = os.path.join(os.path.dirname(DATASET_ROOT), "runs", "yolo_tubo_v3_final")
model = YOLO(os.path.join(RUN_DIR, "weights", "best.pt"))

metrics = model.val()   # data.yaml ya apunta al conjunto de validacion

print(f"mAP50-95: {metrics.box.map}")
print(f"mAP50:    {metrics.box.map50}")
print(f"mAP75:    {metrics.box.map75}")

### 9.1. Evaluacion en el conjunto de prueba (test)

El test (10 %) no se usa ni para entrenar ni para elegir hiperparametros, asi
que es la metrica mas honesta del rendimiento. Desde la regeneracion con
`save_hard_samples:=false` no contiene muestras con el tubo principal visible
menos del 10 % (se descartan al generar y no van sin etiqueta a test).

In [ ]:
from ultralytics import YOLO
RUN_DIR = os.path.join(os.path.dirname(DATASET_ROOT), "runs", "yolo_tubo_v3_final")
model = YOLO(os.path.join(RUN_DIR, "weights", "best.pt"))

test_metrics = model.val(split="test")

print("Resultados en el test set:")
print(f"mAP50-95: {test_metrics.box.map}")
print(f"mAP50:    {test_metrics.box.map50}")
print(f"mAP75:    {test_metrics.box.map75}")

## 10. Predicciones de ejemplo

Inferencia sobre unas imagenes del test set para inspeccionar visualmente las
detecciones (cajas + confianza).

In [ ]:
import os
from IPython.display import Image, display
from ultralytics import YOLO
RUN_DIR = os.path.join(os.path.dirname(DATASET_ROOT), "runs", "yolo_tubo_v3_final")
model = YOLO(os.path.join(RUN_DIR, "weights", "best.pt"))

test_images_path = os.path.join(DATASET_ROOT, "images", "test")
exts = (".png", ".jpg", ".jpeg", ".bmp")
image_files = sorted(
    os.path.join(test_images_path, f)
    for f in os.listdir(test_images_path) if f.lower().endswith(exts)
)

for img_path in image_files[:5]:
    print(f"Prediciendo en: {img_path}")
    res = model.predict(source=img_path, save=True, conf=0.25)
    pred_path = os.path.join(res[0].save_dir, os.path.splitext(os.path.basename(img_path))[0] + ".jpg")
    display(Image(filename=pred_path))
    print("\n---\n")

**Como leer las predicciones**

- **Bounding boxes**: deben rodear el tubo de forma ajustada.
- **Confianza**: score 0-1; sube `conf` (p.ej. 0.5) para menos falsos positivos.
- **Falsos positivos**: cajas donde no hay tubo (ojo con las rocas alargadas).
- **Falsos negativos**: tubos reales sin caja (criticos si el detector alimenta
  al estimador de pose).

## 11. Descargar resultados

Comprime y descarga la carpeta `runs` (pesos `best.pt`, curvas, predicciones).

In [ ]:
from google.colab import files

!zip -r -q runs.zip "{RUN_DIR}"
files.download("runs.zip")